In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from vetuner.correction import gated_correction
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import apply_sensor_model
from vetuner.transport import (apply_variable_delay,compensate_delay,estimate_delay_from_log,exhaust_transport_delay,)
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table
from dataclasses import replace

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

sim = simulate_engine(train, base)

true_delay = exhaust_transport_delay(
    sim.ve_true, sim.rpm, sim.map_kpa, sim.iat_c,
    np.nan_to_num(sim.afr_actual, nan=22.0),
)

In [ ]:
"""
Estimate the transport delay in the exhaust system.
"""

print(f"delay range: {true_delay.min() * 1e3:.1f} to {true_delay.max() * 1e3:.1f} ms")
print(f"median: {np.median(true_delay) * 1e3:.1f} ms")
print(f"sample interval: {(sim.time_s[1] - sim.time_s[0]) * 1e3:.0f} ms")
print(f"fraction of samples with sub-sample delay: "
      f"{(true_delay < sim.time_s[1] - sim.time_s[0]).mean():.0%}")

In [ ]:
"""
Apply variable delay to the AFR signal and simulate the effect on the engine model.
"""

delayed_afr = apply_variable_delay(
    np.nan_to_num(sim.afr_actual, nan=22.0), sim.time_s, true_delay
)
delayed_sim = replace(delayed_sim := sim, afr_actual=delayed_afr)

log_ideal = apply_sensor_model(sim, seed=0)
log_delayed = apply_sensor_model(delayed_sim, seed=0)

estimated = estimate_delay_from_log(log_delayed, base)
log_compensated = compensate_delay(log_delayed, estimated)

print(f"{'case':26s} {'RMSE':>7} {'visited':>8} {'AFR%':>7} {'cover':>6}")
for name, current in [
    ("no delay (reference)", log_ideal),
    ("delayed, uncompensated", log_delayed),
    ("delayed, compensated", log_compensated),
]:
    result, gate = gated_correction(current, base)
    t = evaluate_table(result.table, truth, result.counts)
    a = evaluate_afr(result.table, test)
    print(f"{name:26s} {t.rmse:7.2f} {t.rmse_visited:8.2f} {a.mean_abs_pct:7.2f} "
          f"{t.coverage:6.0%}")

In [ ]:
"""
Plot the true and estimated transport delay across the drive cycle.
"""


fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(sim.time_s, sim.rpm, lw=1.0, color="tab:blue")
axes[0].set_ylabel("engine speed (rpm)")

axes[1].plot(sim.time_s, true_delay * 1e3, lw=1.0, color="tab:red", label="true")
axes[1].plot(log_delayed.time_s, estimated * 1e3, lw=0.8, color="tab:green",
             alpha=0.7, label="estimated from log")
axes[1].axhline((sim.time_s[1] - sim.time_s[0]) * 1e3, color="black", ls="--",
                lw=1.0, label="sample interval")
axes[1].set_ylabel("transport delay (ms)")
axes[1].set_xlabel("time (s)")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Exhaust transport delay across the drive cycle")
fig.tight_layout()

In [ ]:
from dataclasses import replace as dc_replace
from vetuner.gating import GatingConfig

for displacement in (0.2, 0.5, 1.0, 2.0):
    cfg = dc_replace(GatingConfig(), max_cell_displacement=displacement)
    for name, current in [("uncomp", log_delayed), ("comp", log_compensated)]:
        result, gate = gated_correction(current, base, gating=cfg)
        t = evaluate_table(result.table, truth, result.counts)
        a = evaluate_afr(result.table, test)
        print(f"displ {displacement:4.1f} {name:8s} accept {gate.acceptance_rate:4.0%} "
              f"cover {t.coverage:4.0%} visited {t.rmse_visited:5.2f} AFR {a.mean_abs_pct:5.2f}%")